# Behavioral Churn Prediction Framework — Official Experiment Runner

This notebook is the **single official runner** for the complete research
protocol. It executes the full experiment matrix and all post-experiment
analyses (statistical comparison, framework quadrant, model persistence,
publication figures, integrity audit and completion report) using the
framework in `src/`.

## 1. Overview

Sections:

1. Overview
2. Configuration
3. Environment & Reproducibility
4. Dataset Registry — registered / available / validated / executed
5. Experiment Matrix Definition
6. Run Experiments
7. Test-Identity Validation
8. Master Tables & Dataset Characteristics
9. Statistical Comparison (Friedman / Nemenyi / Wilcoxon + CD)
10. Framework Quadrant Analysis
11. Best-Model Persistence & Verification
12. Publication Figures — Master
13. Publication Figures — Supplementary
14. Integrity Audit
15. Completion Report & Conclusions
16. Reproduction Notes

Scientific-integrity guarantees: the framework never tunes the quadrant or
models to predictive performance, and negative SMOTE effects are shown.


In [ ]:
# ── Import bootstrap ──────────────────────────────────────────
# Makes the framework `src` package importable on Kaggle and locally.
# - Local:  searches CWD + ancestors for the repo root (src/ + pyproject.toml).
# - Kaggle: searches every attached dataset mount for pyproject.toml whose
#           parent also contains src/ (e.g. .../final-pipeline/).
import sys, os
from pathlib import Path

def _find_framework_root():
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "src").is_dir() and (cand / "pyproject.toml").exists():
            return cand
    inp = Path('/kaggle/input')
    if inp.is_dir():
        for p in inp.rglob('pyproject.toml'):
            if (p.parent / 'src').is_dir():
                return p.parent
    return None

_PROJECT_ROOT = _find_framework_root()
if _PROJECT_ROOT is None:
    raise SystemExit(
        "Could not locate the framework `src` package. Run from the repo "
        "root or attach the framework dataset on Kaggle."
    )
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))
print(f"framework root : {_PROJECT_ROOT}")


## 2. Configuration

All raw data are supplied on Kaggle. The `DATA_DIRS` mapping below points
each dataset at its mounted input directory, and the individual file-path
constants document the exact Kaggle files each adapter expects. Verify the
mounts before running.


In [ ]:
# ── Core flags ─────────────────────────────────────────────
SMOKE_TEST = True          # True = fast isolated run, never contaminates final results
RUN_EXPERIMENTS = True     # False keeps this as an orchestration-only (analysis) run

# ── Dataset selection ──────────────────────────────────────
# None = the registered FINAL_EXPERIMENT_DATASETS (olist, retailrocket,
# rees46, instacart, telco, online_retail_ii, lastfm, credit_card)
DATASETS = None

# ── Kaggle DATA_DIRS ───────────────────────────────────────
# Mounted dataset directories. Do not invent paths — verify the mounts.
DATA_DIRS = {
    "olist":            "/kaggle/input/datasets/olistbr/brazilian-ecommerce",
    "retailrocket":     "/kaggle/input/datasets/retailrocket/ecommerce-dataset",
    "rees46":           "/kaggle/input/datasets/rees46/rees46-marketplace",
    "instacart":        "/kaggle/input/datasets/psparks/instacart-market-basket-analysis",
    "telco":            "/kaggle/input/datasets/blastchar/telco-customer-churn",
    "online_retail_ii": "/kaggle/input/datasets/nikhilwankhedee/online-retail-ii",
    "lastfm":           "/kaggle/input/datasets/nikhilwankhedee/lastfm",
    "credit_card":      "/kaggle/input/datasets/sakshigoyal7/credit-card-customers",
}

# ── Individual file loading paths (Kaggle) ─────────────────
RETAILROCKET_EVENTS = "/kaggle/input/datasets/retailrocket/ecommerce-dataset/events.csv"
REES46_FILE = "/kaggle/input/datasets/rees46/rees46-marketplace/events.csv"
INSTACART_ORDERS = "/kaggle/input/datasets/psparks/instacart-market-basket-analysis/orders.csv"
INSTACART_PRIOR = "/kaggle/input/datasets/psparks/instacart-market-basket-analysis/order_products__prior.csv"
TELCO_FILE = "/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"
ONLINE_RETAIL_FILE = "/kaggle/input/datasets/nikhilwankhedee/online-retail-ii/online_retail_II.xlsx"
LASTFM_PARQUET = "/kaggle/input/datasets/nikhilwankhedee/lastfm/lastfm-dataset-1k.snappy.parquet"
LASTFM_PROFILE = "/kaggle/input/datasets/nikhilwankhedee/lastfm/userid-profile.tsv"
CREDIT_CARD_FILE = "/kaggle/input/datasets/sakshigoyal7/credit-card-customers/BankChurners.csv"

# ── Model / window overrides ───────────────────────────────
MODEL_NAMES = None          # None = final 5 (LR, RF, XGB, LGBM, SVM)
CHURN_WINDOW_OVERRIDE = None
OUTPUT_DIR = None           # None = project_root/outputs

if SMOKE_TEST:
    # Smoke mode: a tiny, fast, *isolated* run (runner redirects all output
    # to outputs/smoke_test_<timestamp> and marks the dir as SMOKE_TEST).
    DATASETS = ['olist']
    RUN_EXPERIMENTS = True
    print("SMOKE_TEST mode ON — running only", DATASETS,
          "into an isolated output directory.")


## 3. Environment & Reproducibility

Pin and print the exact library versions so the run can be reproduced.


In [ ]:
import sys, os, json, datetime, platform
import numpy as np, pandas as pd

from src.config import (
    PROJECT_ROOT, RANDOM_SEED, FRAMEWORK_VERSION,
    FINAL_EXPERIMENT_DATASETS, FINAL_EXPERIMENT_MODELS, SMOTE_CONDITIONS,
)
from src.utils import get_logger, set_seed
from src.experiment_runner import (
    collect_system_info, run_all_experiments, run_post_processing,
    validate_test_identity, generate_all_results,
)

set_seed(RANDOM_SEED)
print(f"framework version : {FRAMEWORK_VERSION}")
print(f"random seed       : {RANDOM_SEED}")
print(f"registered models : {FINAL_EXPERIMENT_MODELS}")
print(f"smote conditions  : {SMOTE_CONDITIONS}")
print(json.dumps(collect_system_info().get('library_versions', {}), indent=2))


## 4. Dataset Registry — registered / available / validated / executed

`registered` = adapter known to the framework · `available` = raw data
present on disk · `validated` = passed dataset validation · `executed` =
present in the master results.


In [ ]:
from src.datasets import list_datasets, get_dataset

registered = list_datasets()
# A dataset is "available" only if its adapter can actually load non-empty
# raw data (missing files raise / return empty -> False).
def dataset_available(name, data_dirs=None):
    try:
        adapter = get_dataset(name, data_dir=(data_dirs or {}).get(name))
        df = adapter.load_raw_data()
        return bool(df is not None and len(df) > 0)
    except Exception:
        return False

status_rows = []
for name in sorted(registered):
    status_rows.append({
        'dataset': name,
        'registered': True,
        'available': dataset_available(name, DATA_DIRS),
    })
status_df = pd.DataFrame(status_rows)
print("Registered datasets:", len(status_df))
print(status_df.to_string(index=False))

# validated / executed only make sense after a run — populated later in
# Sections 8 & 14 from the generated validation report and master results.


## 5. Experiment Matrix Definition

Every dataset that is registered **and** has data on disk runs in both SMOTE
conditions with all five final models — i.e. `#datasets × 2 × 5` model cells
(80 cells when all 8 datasets are available and validated).


In [ ]:
from src.experiment_runner import validate_datasets

reg = sorted(list_datasets())
if DATASETS is None:
    DATASETS = list(FINAL_EXPERIMENT_DATASETS)

validation_report = validate_datasets(DATASETS, DATA_DIRS)
print(validation_report.to_string(index=False))

valid_datasets = validation_report[validation_report['valid']]['dataset'].tolist()
invalid_datasets = validation_report[~validation_report['valid']]['dataset'].tolist()
print(f"\nValid datasets : {valid_datasets}")
print(f"Invalid        : {invalid_datasets}")

n_expected_cells = len(valid_datasets) * 2 * len(FINAL_EXPERIMENT_MODELS)
print(f"Expected model cells: {n_expected_cells} "
      f"({len(valid_datasets)} datasets × 2 SMOTE × {len(FINAL_EXPERIMENT_MODELS)} models)")
print("(80 cells when all 8 datasets are available and validated)")


## 6. Run Experiments

This cell runs the full matrix plus all post-experiment stages via
`run_all_experiments`. In `SMOKE_TEST` mode the output directory is isolated
(`outputs/smoke_test_<timestamp>`, marked `SMOKE_TEST.txt`) and can never
contaminate the final results.


In [ ]:
%%time
experiment_dir = run_all_experiments(
    datasets=DATASETS,
    data_dirs=DATA_DIRS,
    output_dir=OUTPUT_DIR,
    churn_window_override=CHURN_WINDOW_OVERRIDE,
    model_names=MODEL_NAMES,
    smoke_test=SMOKE_TEST,
)
print("\nExperiment directory:", experiment_dir)
print("Is smoke test     :", SMOKE_TEST)


## 7. Test-Identity Validation

For every dataset the SMOTE and non-SMOTE conditions must share the exact
same test customer set and labels (`test_ids_hash`, `test_y_hash`).


In [ ]:
# Test identity is verified inside the runner: for every dataset the SMOTE
# and non-SMOTE conditions must share the exact same test customer set and
# labels. A failure is recorded in failure_report.csv (check=test_identity);
# absence of a failure for a dataset == PASS.
import os
import pandas as pd

all_results = pd.read_csv(
    os.path.join(experiment_dir, 'results', 'master', 'all_results.csv'))

exp_log = pd.read_csv(
    os.path.join(experiment_dir, 'results', 'master', 'experiment_log.csv'))
print(exp_log[['dataset', 'model', 'smote', 'status',
               'duration_seconds']].to_string(index=False))

fr_path = os.path.join(experiment_dir, 'results', 'master', 'failure_report.csv')
failures = pd.read_csv(fr_path) if os.path.exists(fr_path) else pd.DataFrame()
idf = failures[failures['check'] == 'test_identity'] if len(failures) else failures

identity_rows = []
for ds in sorted(all_results['dataset'].unique()):
    f = idf[idf['dataset'] == ds]
    if len(f):
        identity_rows.append({'dataset': ds, 'valid': False,
                              'note': f['error'].iloc[0]})
    else:
        identity_rows.append({'dataset': ds, 'valid': True,
                              'note': 'PASS — shared test set across SMOTE conditions'})
print("\nTest-identity per dataset:")
print(pd.DataFrame(identity_rows).to_string(index=False))

identity_results = [
    {'dataset': r['dataset'], 'test_ids_match': bool(r['valid']),
     'test_y_match': bool(r['valid']), 'valid': bool(r['valid']),
     'note': r['note']}
    for r in identity_rows]


## 8. Master Tables & Dataset Characteristics

All master tables (all_results, dataset_summary, model_summary,
overall_ranking, smote_comparison, experiment_log, research_summary,
dataset_characteristics) are written to `results/master/`.


In [ ]:
import glob
master_dir = os.path.join(experiment_dir, 'results', 'master')
print("Master tables:")
for f in sorted(glob.glob(os.path.join(master_dir, '*.csv'))):
    print(f"  {os.path.basename(f):<35} {pd.read_csv(f).shape}")

all_results.head()


## 9. Statistical Comparison

Runs after all experiments on the master results: **Friedman** (global model
difference, per SMOTE condition and pooled), **Nemenyi** post-hoc with the
critical-difference diagram, and **paired Wilcoxon** of `with_smote` vs
`without_smote` per model (pairing = `(dataset, model)`), with raw p-values
always reported alongside BH-FDR q-values.


In [ ]:
from src.statistical_comparison import run_statistical_comparison

stats = run_statistical_comparison(experiment_dir, all_results)
for cond, f in stats.get('friedman', {}).items():
    print(f"Friedman [{cond}]  chi2={f['chi2']:.3f}  p={f['p_value']:.4f}  "
          f"significant@0.05={'yes' if f['significant_at_005'] else 'no'}")

w = stats.get('wilcoxon')
if w is not None and not w.empty:
    print("\nPaired Wilcoxon (with_smote vs without_smote per model):")
    print(w.to_string(index=False))


## 10. Framework Quadrant Analysis

Separate post-experiment analysis. The quadrant is fixed from raw-data
characteristics **only** (never from predictive performance); model results
are overlaid afterwards.


In [ ]:
from src.framework_analysis import run_framework_analysis, score_datasets

fw = run_framework_analysis(experiment_dir, all_results)
if fw:
    print("Dataset positions:")
    print(fw['positions'].to_string(index=False))
    print("\nQuadrant performance overlay (descriptive only):")
    print(fw['quadrant_performance'].to_string(index=False))


## 11. Best-Model Persistence & Verification

Best model per dataset × SMOTE condition (by ROC-AUC, baselines excluded) is
persisted as `best_model.pkl` + `best_model_metadata.json`, then reloaded and
verified: re-computed predictions and ROC-AUC must reproduce the master
results exactly.


In [ ]:
from src.model_persistence import persist_best_models, verify_best_models

persist_best_models(experiment_dir, all_results)
ver = verify_best_models(experiment_dir)
print(ver[['dataset', 'condition', 'model', 'status', 'proba_match',
           'expected_roc_auc', 'recomputed_roc_auc']].to_string(index=False))
print("\nOverall model persistence:", ver['status'].mode().iloc[0])


## 12. Publication Figures — Master

Model × dataset ROC-AUC heatmaps, dataset performance, SMOTE effect (negative
effects shown explicitly), model ranking, metric distributions. PNG 300 DPI +
PDF in `results/figures/main/`.


In [ ]:
from src.publication_figures import generate_master_figures

figs = generate_master_figures(experiment_dir, all_results)
print(f"Master figures written: {len(figs)}")
for f in sorted(figs):
    print("  ", os.path.relpath(f, experiment_dir))


## 13. Publication Figures — Supplementary

Per dataset × SMOTE condition: ROC, PR, calibration, confusion matrices,
feature importance (top 20) and SHAP summary (skipped gracefully with a
reason when it cannot run). Written to `results/figures/supplementary/`.


In [ ]:
from src.publication_figures import generate_condition_figures

figs = generate_condition_figures(experiment_dir)
print(f"Supplementary figures written: {len(figs)}")


## 14. Integrity Audit

Verifies the research matrix: expected cell counts, complete coverage (no
missing/duplicate model cells), no NaN/Inf in metrics, test-identity, LightGBM
presence, and model-set sanity.


In [ ]:
from src.final_audit import audit_results, overall_audit_status

# identity_results comes from Section 7 (derived from failure_report.csv).
audit = audit_results(experiment_dir, all_results, identity_results)
print(audit[['check', 'status', 'detail']].to_string(index=False))
print("\nOVERALL AUDIT:", overall_audit_status(audit))


## 15. Completion Report & Conclusions

The final `completion_report.{csv,json,txt}` and the terminal summary block
reflect the audit outcome; nothing is ever marked complete if any integrity
check fails.


In [ ]:
from src.final_audit import write_completion_report

write_completion_report(
    experiment_dir, audit, all_results,
    extras={'successful': int((exp_log['status'] == 'success').sum()),
            'failed': int((exp_log['status'] != 'success').sum())})
print("\nCompletion report written to:", os.path.join(experiment_dir,
      'results', 'master', 'completion_report.csv'))


## 16. Reproduction Notes

- **Smoke mode**: set `SMOKE_TEST = True` — outputs go to an isolated
  `outputs/smoke_test_*` directory and are marked, so the final results are
  never contaminated.
- **Full run**: set `SMOKE_TEST = False`, mount the Kaggle data (see
  `DATA_DIRS`), set `DATASETS = None`.
- **Determinism**: global random seed `RANDOM_SEED = 42`; per-experiment
  seeds are derived deterministically from it.
- **Integrity**: the integrity audit and completion report at
  `results/master/integrity_audit.csv` and
  `results/master/completion_report.{csv,json}` are the source of truth for
  whether the run is complete.
